In [ ]:
# -*- coding: utf-8 -*-

# # 可转债 API 可用性验证## 这个 notebook 只做一件事：确认聚宽研究环境里能不能取到可转债需要的数据。# 跑完后把输出给我，如果某个 cell 报错也原样给我。

In [ ]:
from jqdata import *import numpy as npimport pandas as pdprint('=== 测试 1: get_all_securities 能不能拿到可转债列表 ===')# 尝试 cb 类型 —— 如果报错说明不支持这个类型名try:    cbs = get_all_securities(types=['cb'])    print('types=["cb"] 成功, 数量 %d' % len(cbs))    print('列名:', list(cbs.columns))    print('前5行:')    print(cbs.head())except Exception as e:    print('types=["cb"] 失败: %s' % e)# 备用: 尝试用 stock 类型, 看能不能筛出可转债try:    all_stocks = get_all_securities(types=['stock'])    print('\n所有stock类型数量: %d' % len(all_stocks))    # 可转债深市代码12xxxx, 沪市11xxxx    cb_mask = all_stocks.index.str.match(r'^(12\d{4}\.XSHE|11\d{4}\.XSHG)')    print('符合可转债代码格式的: %d' % cb_mask.sum())    if cb_mask.sum() > 0:        print('前10个:')        print(all_stocks[cb_mask].head(10))except Exception as e:    print('stock类型查询失败: %s' % e)

In [ ]:
print('\n=== 测试 2: get_price 能不能取可转债行情 ===')# 拿一个已知的可转债试试TEST_CB = '113021.XSHG'  # 中信转债 (上海市场, 代码11开头)try:    px = get_price(TEST_CB, start_date='2024-01-01', end_date='2024-01-31',                   frequency='daily', fields=['close', 'volume', 'high', 'low'], fq='pre')    print('get_price 成功')    print('shape:', px.shape)    print('columns:', list(px.columns))    print(px.head())except Exception as e:    print('get_price 失败: %s' % e)# 深市可转债TEST_CB_SZ = '123456.XSHE'  # 随便试一个深市代码try:    px2 = get_price(TEST_CB_SZ, start_date='2024-01-01', end_date='2024-01-31',                    frequency='daily', fields=['close'], fq='pre')    print('深市 get_price 成功, shape:', px2.shape)except Exception as e:    print('深市 %s 失败: %s' % (TEST_CB_SZ, e))

In [ ]:
print('\n=== 测试 3: 能不能拿到可转债专属数据 ===')print('3a. 试试 get_security_info')try:    info = get_security_info(TEST_CB)    print('  display_name:', info.display_name)    print('  name:', info.name)    print('  start_date:', info.start_date)    print('  end_date:', info.end_date)    print('  type:', info.type)except Exception as e:    print('  失败: %s' % e)print('\n3b. 试试 finance.run_query 查可转债基本信息')# 聚宽的可转债数据可能在 BALANCE.CONVERTIBLE_BOND 或类似表中try:    q = query(finance.CONVERTIBLE_BOND_CUR_STATISTICS)    df = finance.run_query(q)    print('  CONVERTIBLE_BOND_CUR_STATISTICS 成功, 行数 %d' % len(df))    print('  列名:', list(df.columns))    print('  前3行:')    print(df.head(3))except Exception as e:    print('  CONVERTIBLE_BOND_CUR_STATISTICS 失败: %s' % e)# 试试看有哪些可转债相关的表print('\n3c. 探索 finance 下可转债相关的表')import financecb_tables = [t for t in dir(finance) if 'BOND' in t.upper() or 'CONVERT' in t.upper()]print('  含BOND/CONVERT的表:')for t in cb_tables:    print('    finance.%s' % t)

In [ ]:
print('\n=== 测试 4: 拿一只可转债的完整日线 ===')# 用全部历史确认数据范围try:    full_px = get_price(TEST_CB, start_date='2018-01-01', end_date='2026-07-29',                        frequency='daily', fields=['close', 'volume'], fq='pre')    print('数据范围: %s ~ %s, 共 %d 条'          % (full_px.index[0].date(), full_px.index[-1].date(), len(full_px)))except Exception as e:    print('失败: %s' % e)

In [ ]:
print('\n=== 测试 5: get_current_data 能拿到的属性 ===')# 用 attribute_history 近5天确认数据新鲜度try:    recent = attribute_history(TEST_CB, 5, '1d', ['close', 'volume', 'high', 'low'])    print('最近5天数据:')    print(recent)except Exception as e:    print('失败: %s' % e)print('\n===== 验证完成 =====')print('请把以上全部输出发给我。')print('如果测试3的 finance 表不存在, 我会用价格+正股价计算转股溢价率。')